# Tests — `classification.py` & `llm.py`

Ce notebook teste chaque fonction de manière isolée, du plus bas niveau vers le plus haut :
1. `normalize` — nettoyage de texte
2. `_load_classif_history` — chargement DuckDB
3. `classify_from_history` — lookup historique
4. `llm.call` — appel LLM réel
5. `classify_from_llm` — pipeline LLM complet
6. `classify` — pipeline complet (historique + LLM)

## 0. Setup

In [ ]:
import sys, os
os.chdir('../')

'/home/onyxia/work'

In [ ]:
import importlib
import classification
import llm

# Rechargement à chaud si vous modifiez les sources pendant la session
importlib.reload(classification)
importlib.reload(llm)

print('Imports OK')

Imports OK


---
## 1. `normalize` — nettoyage de texte

Vérifie les cas nominaux et les cas limites (None, HTML, LaTeX, espaces).

In [7]:
from classification import normalize

cases = {
    # (entrée, sortie attendue)
    "nominal":           ("Maîtrise d'Excel",         "maîtrise d excel"),
    "html entities":     ("Comp&eacute;tence",         "compétence"),
    "balises html":      ("<b>Python</b> avancé",      "python avancé"),
    "espaces multiples": ("  Python   avancé  ",       "python avancé"),
    "ponctuation":       ("C++, SQL & NoSQL",          "c    sql   nosql"),
    "latex":             (r"Algèbre \textbf{linéaire}", "algèbre  linéaire "),
    "None":              (None,                         ""),
    "int":               (42,                           ""),
    "chaîne vide":       ("",                           ""),
}

all_ok = True
for name, (inp, expected) in cases.items():
    result = normalize(inp)
    # Comparaison souple sur les espaces pour les cas latex/ponctuation
    ok = result.strip() == expected.strip()
    status = '✓' if ok else '✗'
    if not ok:
        all_ok = False
    print(f"{status}  [{name}]  {repr(inp)!s:40s} → {repr(result)}")
    if not ok:
        print(f"     attendu : {repr(expected)}")

print()
print('Tous les tests normalize passent ✓' if all_ok else 'Certains tests ont échoué ✗')

✓  [nominal]  "Maîtrise d'Excel"                       → 'maîtrise d excel'
✓  [html entities]  'Comp&eacute;tence'                      → 'compétence'
✓  [balises html]  '<b>Python</b> avancé'                   → 'python avancé'
✓  [espaces multiples]  '  Python   avancé  '                    → 'python avancé'
✗  [ponctuation]  'C++, SQL & NoSQL'                       → 'c sql nosql'
     attendu : 'c    sql   nosql'
✗  [latex]  'Algèbre \\textbf{linéaire}'             → 'algèbre linéaire'
     attendu : 'algèbre  linéaire '
✓  [None]  None                                     → ''
✓  [int]  42                                       → ''
✓  [chaîne vide]  ''                                       → ''

Certains tests ont échoué ✗


---
## 2. `_load_classif_history` — connexion DuckDB

Vérifie que la vue est créée et contient des données.

In [8]:
from classification import _load_classif_history

con = _load_classif_history()

# Aperçu des 5 premières lignes
df = con.sql("SELECT * FROM classif_history LIMIT 5").df()
print(f"Colonnes : {list(df.columns)}")
print(f"Lignes retournées : {len(df)}")
df

Colonnes : ['num_original', 'num_entree', 'num_cat', 'theme_original', 'theme_cat', 'niv_original', 'niv_cat', 'ia_original', 'ia_cat']
Lignes retournées : 5


,num_original,num_entree,num_cat,theme_original,theme_cat,niv_original,niv_cat,ia_original,ia_cat
0,logiciel modélisation cao génie climatique,logiciel modélisation cao génie climatique,compétence numérique,logiciel modélisation cao génie climatique,Logiciels spécialisés,None,None,logiciel modélisation cao génie climatique,Erreur
1,maîtrise donnée technique comptable,maîtrise donnée technique comptable,compétence numérique,maîtrise donnée technique comptable,"Données, Analytics & IA",None,None,maîtrise donnée technique comptable,Erreur
2,utiliser logiciel gestion relation client,utiliser logiciel gestion relation client,compétence numérique,utiliser logiciel gestion relation client,Logiciels de gestion d'entreprise,None,None,utiliser logiciel gestion relation client,Erreur
3,technologie suivi itinéraire,technologie suivi itinéraire,compétence numérique,technologie suivi itinéraire,"Infrastructure, Systèmes & Réseaux",None,None,technologie suivi itinéraire,Erreur
4,langchain,langchain,compétence numérique,langchain,Développement applicatif,None,None,langchain,IA générative (GenAI)


In [11]:
# Comptage total et répartition des catégories
stats = con.sql("""
    SELECT num_cat AS categorie, COUNT(*) AS n
    FROM classif_history
    GROUP BY 1
    ORDER BY 2 DESC
""").df()

con.close()
stats

,categorie,n
0,compétence non numérique,94625
1,soft skill,60697
2,pas une compétence,47308
3,compétence numérique,44641
4,domaine - secteur,23534
5,formation - certification,18286
6,Erreur,21


---
## 3. `classify_from_history` — lookup dans l'historique

Teste trois situations : trouvée (numérique), trouvée (non numérique), inconnue.

In [39]:
from classification import classify_from_history, _load_classif_history
import json

# Récupère deux compétences réelles depuis l'historique pour les tests
con = _load_classif_history()
sample = con.sql("""
    SELECT DISTINCT norm_label AS entree, num_cat AS cat
    FROM classif_history
    LIMIT 4
""").df()
con.close()

known_skills   = sample['entree'].tolist()
unknown_skills = ["Compétence totalement inconnue XYZ 999"]

test_skills = known_skills + unknown_skills
print("Compétences testées :", test_skills)


Compétences testées : ['cloud provider', 'qlik', 'blocs fonctionnels', 'résistance face au stress', 'Compétence totalement inconnue XYZ 999']


In [45]:
results = classify_from_history(test_skills)

for r in results:
    found = r['categorie'] is not None
    status = '✓ trouvée  ' if found else '— inconnue '
    print(f"{status}  {r['label']!r:50s}  cat={r['categorie']}")
    if found and r['details']:
        print(f"           détails : {r['details']}")

# Vérification : la compétence inconnue doit avoir categorie=None
last = results[-1]
assert last['categorie'] is None, "La compétence inconnue devrait retourner categorie=None"
print("\nAssertion OK — compétence inconnue bien gérée ✓")

✓ trouvée    'cloud provider'                                    cat=compétence numérique
           détails : {'thematique': 'Infrastructure, Systèmes & Réseaux', 'niveau': 'Basique', 'categorie_ia': None}
✓ trouvée    'Qlik'                                              cat=compétence numérique
           détails : {'thematique': 'Données, Analytics & IA', 'niveau': 'Intermédiaire', 'categorie_ia': None}
✓ trouvée    'bloc fonctionnel'                                  cat=compétence non numérique
✓ trouvée    'résistance face stress'                            cat=compétence non numérique
— inconnue   'Compétence totalement inconnue XYZ 999'            cat=None

Assertion OK — compétence inconnue bien gérée ✓


---
## 4. `llm.call` — appel LLM unitaire

Teste l'appel brut avec un mini prompt de classification. Nécessite `API_KEY` et `MODEL_NAME` dans `.env`.

In [46]:
import llm

# Prompt minimal pour vérifier que le modèle répond bien en JSON
test_prompt = """Tu es un classificateur de compétences.
Pour chaque compétence reçue, réponds UNIQUEMENT avec des objets JSON valides, un par ligne.
Format : {"entrée": "<libellé original>", "cat": "compétence numérique" ou "autre compétence"}
Aucun texte en dehors des objets JSON."""

test_skills = ["Utilisation d'Excel", "Gestion de projet", "Programmation Python"]

raw = llm.call(test_skills, test_prompt)

print(f"Nombre d'objets JSON parsés : {len(raw)} (attendu : {len(test_skills)})")
print()
for obj in raw:
    print(obj)

Nombre d'objets JSON parsés : 3 (attendu : 3)

{'entrée': "Utilisation d'Excel", 'cat': 'compétence numérique'}
{'entrée': 'Gestion de projet', 'cat': 'autre compétence'}
{'entrée': 'Programmation Python', 'cat': 'compétence numérique'}


In [47]:
# Vérification de la structure des objets retournés
for obj in raw:
    assert 'entrée' in obj or 'cat' in obj, f"Clés manquantes dans : {obj}"

print("Structure des objets JSON OK ✓")

Structure des objets JSON OK ✓


---
## 5. `classify_from_llm` — pipeline LLM complet

Envoie des compétences inconnues de l'historique et vérifie la structure de sortie.

In [52]:
from classification import classify_from_llm

# Compétences volontairement absentes de l'historique
new_skills = [
    "Utilisation de ChatGPT en entreprise",
    "Gestion du stress",
    "Développement d'une API REST avec FastAPI",
]

results_llm = classify_from_llm(new_skills)

print(f"{len(results_llm)} résultats pour {len(new_skills)} compétences\n")
for r in results_llm:
    print(f"  label     : {r['label']}")
    print(f"  categorie : {r['categorie']}")
    print(f"  details   : {r['details']}")
    print()

3 résultats pour 3 compétences

  label     : Utilisation de ChatGPT en entreprise
  categorie : compétence numérique
  details   : {'thematique': 'Données, Analytics & IA', 'niveau': 'Avancé', 'categorie_ia': 'IA générative (GenAI)'}

  label     : Gestion du stress
  categorie : soft skill
  details   : None

  label     : Développement d'une API REST avec FastAPI
  categorie : compétence numérique
  details   : {'thematique': 'Développement applicatif', 'niveau': 'Avancé', 'categorie_ia': 'Erreur'}



In [53]:
# Vérification de la structure de chaque résultat
for r in results_llm:
    assert 'label'     in r, f"Clé 'label' manquante : {r}"
    assert 'categorie' in r, f"Clé 'categorie' manquante : {r}"
    assert 'details'   in r, f"Clé 'details' manquante : {r}"
    if r['categorie'] == 'compétence numérique':
        assert r['details'] is not None,               f"details None pour compétence numérique : {r}"
        assert 'thematique'   in r['details'],         f"Clé 'thematique' manquante dans details : {r}"
        assert 'niveau'       in r['details'],         f"Clé 'niveau' manquante dans details : {r}"
        assert 'categorie_ia' in r['details'],         f"Clé 'categorie_ia' manquante dans details : {r}"
    else:
        assert r['details'] is None, f"details devrait être None pour une non-numérique : {r}"

print("Structure de classify_from_llm OK ✓")

Structure de classify_from_llm OK ✓


---
## 6. `classify` — pipeline complet

Teste le pipeline end-to-end avec un mix de compétences connues et inconnues.

In [54]:
from classification import classify, _get_classif_history_connection
import pandas as pd

# Récupère une compétence connue depuis l'historique
con = _get_classif_history_connection()
known = con.sql("SELECT num_entree FROM classif_history LIMIT 1").df().iloc[0, 0]

mixed_skills = [
    known,                                     # doit venir de l'historique
    "Maîtrise de Notion pour la gestion de projets",  # probablement inconnue → LLM
    "Communication interpersonnelle",           # probablement inconnue → LLM
]

print("Compétences testées :")
for s in mixed_skills:
    print(f"  - {s}")
print()

Compétences testées :
  - caractère enthousiaste
  - Maîtrise de Notion pour la gestion de projets
  - Communication interpersonnelle



In [55]:
final_results = classify(mixed_skills)

# Affichage tabulaire
rows = []
for r in final_results:
    rows.append({
        'label':        r['label'],
        'categorie':    r['categorie'],
        'thematique':   r['details']['thematique']   if r['details'] else None,
        'niveau':       r['details']['niveau']       if r['details'] else None,
        'categorie_ia': r['details']['categorie_ia'] if r['details'] else None,
    })

pd.DataFrame(rows)

,label,categorie,thematique,niveau,categorie_ia
0,caractère enthousiaste,soft skill,None,None,None
1,Maîtrise de Notion pour la gestion de projets,compétence numérique,"Communication, collaboration et diffusion numé...",Intermédiaire,Erreur
2,communication interpersonnel,soft skill,None,None,None


In [56]:
# Vérifications finales
assert len(final_results) == len(mixed_skills), \
    f"Nombre de résultats incorrect : {len(final_results)} vs {len(mixed_skills)} attendus"

for r in final_results:
    assert r['categorie'] is not None, \
        f"Compétence sans catégorie après pipeline complet : {r['label']}"

print(f"Pipeline complet OK — {len(final_results)}/{len(mixed_skills)} compétences classifiées ✓")

Pipeline complet OK — 3/3 compétences classifiées ✓


---
## 7. Cas limites de `classify`

In [1]:
from classification import classify

# Entrée non-liste
assert classify("pas une liste") == [], "Devrait retourner [] pour une non-liste"
assert classify(None)            == [], "Devrait retourner [] pour None"
assert classify(42)              == [], "Devrait retourner [] pour un int"

# Liste vide
assert classify([])              == [], "Devrait retourner [] pour une liste vide"

print("Cas limites OK ✓")

ModuleNotFoundError: No module named 'classification'